# Deep Agents Backends — Where Files Are Stored

Every Deep Agent uses a virtual filesystem to read, write, and manage files through paths such as `/notes/todo.txt`. The backend determines where these files are stored and how they persist across runs, threads, and process restarts.

In this notebook, we explore three backends:

- **StateBackend:** Stores files in the agent's state.
- **FilesystemBackend:** Stores files on the local disk.
- **StoreBackend:** Stores files in a LangGraph store under a namespace.

The agent's file operations remain similar, while the backend determines the storage behavior and persistence characteristics.

---

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

---

## 1. StateBackend — Thread-Scoped Files

StateBackend stores files in the agent's LangGraph state rather than directly on disk or in a shared store.

- **Storage:** Agent state, typically maintained in memory during execution.
- **Access:** Files can be accessed through the returned state and carried forward when the state is passed into a subsequent invocation.
- **Persistence:** Files are not independently saved to disk. Their availability across invocations depends on how the agent's state is managed and persisted.
- **Use cases:** Temporary files, intermediate results, and short-lived task context.

In [2]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

### Initialize the Agent with StateBackend

In [ ]:
agent=create_deep_agent(model="groq:openai/gpt-oss-120b")

# Under the hood this is what `agent` is doing - explicit StateBackend
agent2=create_deep_agent(
    model="groq:openai/gpt-oss-120b",
    backend=StateBackend(),
)

### Write a File Using StateBackend
StateBackend keeps that file inside LangGraph State and stored it in RAM, not in folder

In [ ]:
result = agent2.invoke({
    "messages":[{
        "role":"user",
        "content":(
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

### The agent's final natural-language reply
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
The file **/notes/todo.txt** has been created with the requested content. Let me know if you need anything else!


In [7]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='1124a35b-0c41-4951-91ce-479d321f996c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to create file at /notes/todo.txt with given content. Use write_file tool. Then respond.', 'tool_calls': [{'id': 'fc_b461381c-6454-4a83-a407-dfe2c8462194', 'function': {'arguments': '{"content":"1. Record video\\n2. Edit video\\n3. Upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 1938, 'total_tokens': 2013, 'completion_time': 0.158768012, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.078080967, 'prompt_tokens_details': None, 'queue_time': 0.388236237, 'total_time': 0.236848979}, 'model_name': 'openai/gpt-oss-1

### Verify File Creation in Agent State
written files apper under result["files"]

In [ ]:
print("\n--- Backend check -------------------------------------------------")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")


--- Backend check -------------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video', 'encoding': 'utf-8', 'created_at': '2026-10-15T08:10:14.793408+00:00', 'modified_at': '2026-10-15T08:10:14.793408+00:00'}


### Read the File Back Within the Same Thread
feed the returned state back in and ask it to READ the file

In [9]:
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
Here’s the content of **/notes/todo.txt**:

```
1. Record video
2. Edit video
3. Upload video
```


---

## 2. FilesystemBackend — Local Disk Storage

FilesystemBackend stores files on the local filesystem under a configured `root_dir`. With `virtual_mode=True`, virtual paths such as `/notes/todo.txt` map to files within the configured root directory.

- **Storage:** Local disk.
- **Persistence:** Files remain available across agent instances and process restarts, provided the files and directory remain intact.
- **Access:** Files can be inspected and edited using normal filesystem tools.
- **Use cases:** Working with project files, generating reports, and maintaining artifacts across runs.
- **Security:** Use a dedicated working directory and restrict filesystem access to the files the agent needs to manage.

### Initialize the Agent with FilesystemBackend
root_dir="." -> files land relative to your current working directory

virtual_mode=True -> agent uses virtual paths like /notes/todo.txt, mapped onto root_dir

In [10]:
from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(
    model="groq:openai/gpt-oss-120b",
    backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))

print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


### Write a File to the Local Filesystem

In [11]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

### The agent's final natural-language reply
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
The file **/notes/todo.txt** has been created with the requested content.


### Verify File Creation on Disk — look on the REAL disk
With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt

In [12]:
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 D:\agentic-ai\08-Deep-Agents\08-Deep-Agents\notes\todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


### Verify File Persistence Across Agent Instances
Unlike StateBackend, this file survives even after Python exits.

A brand-new agent (fresh state) can read it back from disk.

In [14]:
fresh_agent = create_deep_agent(
    model="groq:openai/gpt-oss-120b",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)


--- Read-back with a FRESH agent (proves disk persistence) ------
1. Record video  
2. Edit video  
3. Upload video


---

## 3. StoreBackend — Namespace-Based Storage

StoreBackend stores files in a LangGraph store under a configured namespace, allowing them to be accessed across different threads that share the same store and namespace.

- **Storage:** LangGraph store.
- **Access:** Multiple threads can access the same stored files when they use the same store and namespace.
- **Persistence:** With `InMemoryStore`, data remains available only while the in-memory store is retained. A persistent store is required for durability across process restarts.
- **Use cases:** Shared user context, reusable information, and files that need to be available across conversations.

### Initialize the Agent with StoreBackend

In [15]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="groq:openai/gpt-oss-120b",
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

✅ Agent created with StoreBackend + static namespace.


### Write a File in Thread 1

In [16]:
import os
import uuid

thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
The file **/notes/todo.txt** has been created with the requested content. Let me know if there’s anything else you’d like to add or modify!


### Read the File from Thread 2

In [17]:
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
1. Record video  
2. Edit video  
3. Upload video


NOTE: With StoreBackend + InMemoryStore, the file is not saved to disk at all. It lives in RAM, inside the InMemoryStore object, as an entry keyed under the namespace.

---

## Backend Comparison and Key Takeaways

| Feature | `StateBackend` | `FilesystemBackend` | `StoreBackend` |
|---|---|---|---|
| **Storage location** | Agent state | Local disk | LangGraph store |
| **Accessible across threads** | Depends on state sharing | Yes, if paths are shared | Yes, with the same store and namespace |
| **Survives process restart** | Depends on state persistence | Yes | Only with a persistent store |
| **Creates real disk files** | No | Yes | No |
| **Best suited for** | Temporary context | Local files and artifacts | Shared, reusable context |

**Mental model:** the agent never knows the difference. Its tools always speak in file paths — the backend quietly translates every read/write into the right operation underneath. That's what lets you swap backends without touching a line of agent logic.